In [1]:
import os
import time
import pandas as pd
import yfinance as yf
import sys
sys.path.append("..")

In [2]:
# ETF universe: related-but-distinct economic exposures.
# Deliberately excludes highly correlated index trackers (no SPY+IVV, no SPY+QQQ).
TICKERS = [
    # Sector ETFs (SPDR)
    'XLF', 'XLE', 'XLK', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC',
    # Sub-sector financials
    'KBE', 'KRE', 'KIE',
    # Sub-sector energy
    'XOP', 'OIH', 'USO',
    # Country ETFs (iShares MSCI)
    'EWJ', 'EWG', 'EWU', 'EWA', 'EWC', 'EWZ', 'EWY', 'EWT', 'EWH',
    # Commodity
    'GLD', 'SLV', 'GDX', 'GDXJ', 'UNG',
    # Style
    'IWM', 'IWD', 'IWF', 'MTUM', 'QUAL',
    # Bond
    'TLT', 'IEF', 'LQD', 'HYG',
]

START_DATE = '2020-01-01'
END_DATE   = '2026-01-01'
RAW_PATH   = '../data_raw/prices.csv'

print(f"Universe: {len(TICKERS)} tickers")

Universe: 40 tickers


In [3]:
def cache_is_valid(path, tickers, start, end):
    """Return True if cached CSV already covers the required universe and date range."""
    if not os.path.exists(path):
        return False
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    missing = set(tickers) - set(df.columns)
    if missing:
        print(f"Cache miss: missing tickers {missing}")
        return False
    if df.index.min() > pd.Timestamp(start) or df.index.max() < pd.Timestamp(end) - pd.Timedelta(days=5):
        print("Cache miss: date range insufficient")
        return False
    return True


def download_with_retry(tickers, start, end, max_retries=3):
    for attempt in range(max_retries):
        try:
            raw = yf.download(tickers, start=start, end=end, interval='1d', auto_adjust=True, progress=True)
            return raw['Close']
        except Exception as e:
            wait = 2 ** attempt
            print(f"Attempt {attempt+1} failed ({e}), retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"Download failed after {max_retries} attempts")


if cache_is_valid(RAW_PATH, TICKERS, START_DATE, END_DATE):
    print("Using cached prices.csv — skipping download.")
    prices = pd.read_csv(RAW_PATH, index_col=0, parse_dates=True)
else:
    print("Downloading fresh price data...")
    prices = download_with_retry(TICKERS, START_DATE, END_DATE)
    prices.index.name = 'Date'

prices.shape

Cache miss: date range insufficient


[*********************100%***********************]  40 of 40 completed


(1508, 40)

In [4]:
# Report data coverage per ticker (flag anything with >5% missing days)
n_total = len(prices)
coverage = 1 - prices.isnull().mean()
poor = coverage[coverage < 0.95]
if len(poor):
    print("Tickers with <95% coverage — consider dropping:")
    print(poor.sort_values())
else:
    print("All tickers have >=95% coverage.")

print(f"\nDate range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Total trading days: {n_total}")

All tickers have >=95% coverage.

Date range: 2020-01-02 to 2025-12-31
Total trading days: 1508


In [5]:
# Drop rows where ALL tickers are missing (weekends/holidays artifact)
prices = prices.dropna(how='all')

prices.to_csv(RAW_PATH)
print(f"Saved {prices.shape[0]} rows x {prices.shape[1]} tickers to {RAW_PATH}")
prices.head(3)

Saved 1508 rows x 40 tickers to ../data_raw/prices.csv


Ticker,EWA,EWC,EWG,EWH,EWJ,EWT,EWU,EWY,EWZ,GDX,...,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY,XOP
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,18.085592,26.467484,25.817459,20.053408,52.426125,27.770473,27.294142,55.563740,31.972643,27.242306,...,23.356087,27.642969,75.150536,44.294178,52.989922,31.014631,26.264919,92.306091,60.155190,81.182480
2020-01-03,17.894716,26.335058,25.324957,19.779804,51.848080,27.370516,27.046381,54.437626,31.695257,27.075230,...,23.286694,27.349466,75.005684,43.796177,52.905121,31.241846,26.318428,91.501709,59.643269,82.452057
2020-01-06,17.990156,26.467484,25.359518,19.836134,52.032001,27.190535,27.214220,54.241005,31.206514,27.121641,...,23.467892,27.331680,75.032829,43.900513,53.015373,31.249968,26.343124,92.071098,59.809170,83.001030
